In [5]:
import os
import shutil

src = "/kaggle/input/datasets/dlcantoine/abc-transformer"

for file in os.listdir(src):
    shutil.copy(os.path.join(src, file), "/kaggle/working/")


In [6]:
os.listdir()

['model.py', '.virtual_documents', 'data_utils.py', 'tokenizer_utils.py']

In [7]:
import torch
import random

from data_utils import load_abc_dataset, train_val_split
from tokenizer_utils import train_tokenizer
from model import ABCTransformer

In [20]:
# Data
MAX_EXAMPLES = 50000
VAL_RATIO = 0.1

# Tokenizer
VOCAB_SIZE = 1024

# Model
BLOCK_SIZE = 256
EMBED_DIM = 256
NUM_HEADS = 4
NUM_LAYERS = 4
DROPOUT = 0.1

# Training
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
MAX_STEPS = 3000
EVAL_INTERVAL = 200
EVAL_ITERS = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [21]:
texts = load_abc_dataset(max_examples=MAX_EXAMPLES)
train_texts, val_texts = train_val_split(texts, val_ratio=VAL_RATIO)

print("Total:", len(texts))
print("Train:", len(train_texts))
print("Val:", len(val_texts))
print()
print(train_texts[0][:500])

Total: 50000
Train: 45000
Val: 5000

X:37530
L:1/8
M:2/4
K:D
|: f/e/ | dAAA | B2 A2 | BdFD | GE Ef/e/ | B2 A2 | Bgfe | fdd :: f/g/ | affd | bgge | affd | 
 e/d/c/B/ Af/g/ | affd | bgge | fa AB/c/ | d2 D :|


In [22]:
tokenizer = train_tokenizer(
    train_texts,
    vocab_size=VOCAB_SIZE,
    save_path="tokenizer.json"
)

print("Tokenizer vocab size:", tokenizer.get_vocab_size())




Tokenizer vocab size: 1024


In [23]:
train_token_ids = []
for text in train_texts:
    ids = tokenizer.encode(text).ids
    if len(ids) > 1:
        train_token_ids.extend(ids + [tokenizer.token_to_id("[EOS]")])

val_token_ids = []
for text in val_texts:
    ids = tokenizer.encode(text).ids
    if len(ids) > 1:
        val_token_ids.extend(ids + [tokenizer.token_to_id("[EOS]")])

train_data = torch.tensor(train_token_ids, dtype=torch.long)
val_data = torch.tensor(val_token_ids, dtype=torch.long)

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))

Train tokens: 7592895
Val tokens: 849550


In [24]:
# Inspect tokenizer and vocabulary

# 1. Vocabulary size
print("Vocabulary size:", tokenizer.get_vocab_size())
print()

# 2. Get vocab dictionary
vocab = tokenizer.get_vocab()

# 3. Sort vocab by token id
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])

print("First 100 tokens in vocabulary:")
print("-" * 40)

for token, idx in sorted_vocab[:100]:
    print(f"{idx:4d}  {repr(token)}")

print("\n")

# 4. Look at tokenizer behavior on a sample
sample = train_texts[0]

print("Sample ABC text:")
print("-" * 40)
print(sample[:400])
print("\n")

encoded = tokenizer.encode(sample)

print("First 50 tokens from tokenizer:")
print("-" * 40)
print(encoded.tokens[:50])
print("\n")

print("First 50 token ids:")
print("-" * 40)
print(encoded.ids[:50])
print("\n")

# 5. Decode to verify reconstruction
decoded = tokenizer.decode(encoded.ids)

print("Decoded text preview:")
print("-" * 40)
print(decoded[:400])
print("\n")

# 6. Show unique tokens used in this sample
unique_tokens = set(encoded.tokens)

print("Unique tokens in this example:")
print("-" * 40)
print(list(unique_tokens)[:50])

Vocabulary size: 1024

First 100 tokens in vocabulary:
----------------------------------------
   0  '[PAD]'
   1  '[UNK]'
   2  '[BOS]'
   3  '[EOS]'
   4  '!'
   5  '"'
   6  '#'
   7  '$'
   8  '&'
   9  "'"
  10  '('
  11  ')'
  12  '*'
  13  '+'
  14  ','
  15  '-'
  16  '.'
  17  '/'
  18  '0'
  19  '1'
  20  '2'
  21  '3'
  22  '4'
  23  '5'
  24  '6'
  25  '7'
  26  '8'
  27  '9'
  28  ':'
  29  ';'
  30  '<'
  31  '='
  32  '>'
  33  '?'
  34  '@'
  35  'A'
  36  'B'
  37  'C'
  38  'D'
  39  'E'
  40  'F'
  41  'G'
  42  'H'
  43  'I'
  44  'J'
  45  'K'
  46  'L'
  47  'M'
  48  'N'
  49  'O'
  50  'P'
  51  'Q'
  52  'R'
  53  'S'
  54  'T'
  55  'U'
  56  'V'
  57  'W'
  58  'X'
  59  'Y'
  60  'Z'
  61  '['
  62  '\\'
  63  ']'
  64  '^'
  65  '_'
  66  '`'
  67  'a'
  68  'b'
  69  'c'
  70  'd'
  71  'e'
  72  'f'
  73  'g'
  74  'h'
  75  'i'
  76  'j'
  77  'k'
  78  'l'
  79  'm'
  80  'n'
  81  'o'
  82  'p'
  83  'q'
  84  'r'
  85  's'
  86  't'
  87  'u'
  88  '

In [25]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [26]:
xb, yb = get_batch("train")
print(xb.shape, yb.shape)
print(xb[0][:20])
print(yb[0][:20])

torch.Size([16, 256]) torch.Size([16, 256])
tensor([118, 353,  99,  19, 342, 835, 116,  20, 342, 113, 243,  68, 126, 635,
        101,  17,  35,  17,  35, 635], device='cuda:0')
tensor([353,  99,  19, 342, 835, 116,  20, 342, 113, 243,  68, 126, 635, 101,
         17,  35,  17,  35, 635, 142], device='cuda:0')


In [27]:
vocab_size = tokenizer.get_vocab_size()

model = ABCTransformer(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

print("Parameters (M):", sum(p.numel() for p in model.parameters()) / 1e6)

Parameters (M): 3.747328


In [28]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [29]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}

    for split in ["train", "val"]:
        split_losses = []

        for _ in range(EVAL_ITERS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            split_losses.append(loss.item())

        losses[split] = sum(split_losses) / len(split_losses)

    model.train()
    return losses

In [30]:
model.train()

for step in range(MAX_STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

step 0: train loss 7.0968, val loss 7.0995
step 200: train loss 3.4021, val loss 3.4013
step 400: train loss 3.0264, val loss 3.0081
step 600: train loss 2.8368, val loss 2.8348
step 800: train loss 2.7404, val loss 2.7178
step 1000: train loss 2.6102, val loss 2.6078
step 1200: train loss 2.5009, val loss 2.5070
step 1400: train loss 2.3964, val loss 2.4221
step 1600: train loss 2.2973, val loss 2.3201
step 1800: train loss 2.2398, val loss 2.2415
step 2000: train loss 2.1912, val loss 2.1947
step 2200: train loss 2.1116, val loss 2.1645
step 2400: train loss 2.0510, val loss 2.1045
step 2600: train loss 2.0420, val loss 2.0392
step 2800: train loss 1.9975, val loss 2.0004


In [32]:
torch.save(model.state_dict(), "abc_transformer.pt")
print("Model saved.")

Model saved.


In [33]:
@torch.no_grad()
def generate_text(prompt="", max_new_tokens=300, temperature=1.0):
    model.eval()

    if prompt.strip():
        start_ids = tokenizer.encode(prompt).ids
    else:
        bos_id = tokenizer.token_to_id("[BOS]")
        if bos_id is None:
            start_ids = [0]
        else:
            start_ids = [bos_id]

    x = torch.tensor([start_ids], dtype=torch.long, device=device)

    y = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature)
    token_ids = y[0].tolist()

    text = tokenizer.decode(token_ids)
    return text

In [35]:
sample = generate_text(
    prompt="X:1\nT:Generated Tune\nM:4/4\nL:1/8\nK:D\n",
    max_new_tokens=180,
    temperature=0.7
)

print(sample)

 X:1
T:Generated Tune
M:4/4
L:1/8
K:D
 d2 d2 (gf) | d2 (gf) | d2 (ed) | d2 (dB) | d2 d2 (Bc) | d2 (BA) | d2 (Bc) | d2 (ed) | d2 d2 (cA) | 
 d2 (Bc) | d3 (ef) | d2 (eg) | (ga) | e2 (ga) | (gf) | a2) | d2 (fd) | d2 (cd) | d2 (df) | d2 (ef) | (gf) | 
 (ga) | (dc) | d2 (Ac) | d2 (ef) | d2 (dB) | d2 (de) | d2 (dB) | d2 (Bg) | d2 (ed) | d2 (ef


In [39]:
for i in range(3):
    print("=" * 80)
    print(f"SAMPLE {i+1}")
    print("=" * 80)
    print(generate_text(
        prompt="X:1\nT:Generated Tune\nM:4/4\nL:1/8\nK:G\n",
        max_new_tokens=220,
        temperature=0.65
    ))
    print()

SAMPLE 1
 X:1
T:Generated Tune
M:4/4
L:1/8
K:G
|: B2 d2 B2 | B2 B2 d2 e2 | e2 d2 B2 | B2 B2 B2 | B2 B2 B2 B2 | G2 B2 B2 e2 | e2 e2 e2 g2 | 
 g2 e2 d2 d2 B2 | B2 B2 d2 B2 | B2 B2 d2 d2 d2 e2 | f2 e2 e2 e2 d2 | B2 G2 B2 | B2 B2 B2 e2 d2 d2 B2 B2 | 
 B2 B2 B2 B2 B2 B2 | B3 B2 B2 | B2 B2 B2 c2 B2 d2 | A2 G2 B2 | B2 B2 B2 d2 B2 B2 c2 B2 B2 | G4 G2 B2 A2 | 
 G6 G2 B2 d2 B2 | G4 G2 B2 B2 B2 | B

SAMPLE 2
 X:1
T:Generated Tune
M:4/4
L:1/8
K:G
 (3G2 | (3G2 G>B B>G | G>AB>G (3G2 G>B | d>ef>g | (3B3 G | A>BA>G (3G2 G>F | B>AB>A (3G2 G>A | 
 A>GA>B B>AB>G (3D | E>GA>B (3G2 G>B | G>B B>g (3G2 G>G | A>AG>G (3G2 G>B | A>GA>B (3G2 G>B | d>ed>e d>G (3G2 G>B | 
 A>BA>B (3G2 F>G | A>GA>B (3G2 G>B | c>AB>g (3g2 G2 | A>Bc>d B>d | e>de>c d>e d>B | 
 c>ed>d B>G

SAMPLE 3
 X:1
T:Generated Tune
M:4/4
L:1/8
K:G
 D | G2 BG | G2 BG | G2 BG | G2 BG | G2 BG | G2 BG | G2 dg | g2 bg | g2 dg | g2 bg | g2 g2 | 
 g2 ge | d2 Bd | g2 dg | g2 dg | g2 dg | g2 dg | g2 dB | G2 :| X:28772
L:1/8
M:4/4
K:D
 D | F2 D2 | F2 A>G | 

In [40]:
import difflib

sample = """
X:1
T:Generated Tune
M:4/4
L:1/8
K:G
 G2 | G2 G>B B>G | G>AB>G G>B | d>ef>g | B3 G | A>BA>G G>F | B>AB>A G>A |
A>GA>B B>AB>G D | E>GA>B G>B | G>B B>g G>G | A>AG>G G>B | A>GA>B G>B | d>ed>e d>G G>B |
A>BA>B F>G | A>GA>B G>B | c>AB>g G2 | A>Bc>d B>d |
""".strip()

scores = []
for i, tune in enumerate(train_texts):
    score = difflib.SequenceMatcher(None, sample, tune.strip()).ratio()
    scores.append((score, i, tune))

scores.sort(reverse=True, key=lambda x: x[0])

for rank, (score, i, tune) in enumerate(scores[:5], start=1):
    print("=" * 80)
    print(f"TOP {rank}")
    print("Similarity:", score)
    print("Index:", i)
    print(tune[:800])
    print()

TOP 1
Similarity: 0.4619164619164619
Index: 21339
X:11982
L:1/4
M:4/4
K:G
 G | G G d d | G F G G | A B c A | B B A A | G A B B | E2 A2 | G F/E/ D A | A E A G | A2 B2 | 
 A G A G | d3 z | B A/B/ A E/F/ | G3 |]

TOP 2
Similarity: 0.4494949494949495
Index: 43375
X:16705
L:1/8
M:3/4
K:G
 BA | G2 GDGA | B2 G2 AB | c2 B2 AG | A4 BA | G2 GDGA | B2 G2 AB | c2 B2 AG | A4 cB | A2 G2 FG | 
 A4 dc | B2 B2 AA | G4 |]

TOP 3
Similarity: 0.42857142857142855
Index: 23369
X:44038
L:1/4
M:3/4
K:G
 d/c/ | B2 B | B A B | d c c | c2 B | B A A | A G A | c B B | B2 d/c/ | B2 B | B A B | d c c | 
 c2 B | B A A | A G A | c B B | B2 |]

TOP 4
Similarity: 0.42727272727272725
Index: 21545
X:14383
L:1/8
M:6/8
K:D
 GBG FAF | E2 B BAB | GBG FAF | D2 A AFD | GBG FAF | E2 B BAB | GAB dBG | ABG FED :| GAB dBd | 
 edB dBA | GAB dBG | ABG FED | GAB dBd | edB def | gfe dBG | ABG FED :|

TOP 5
Similarity: 0.42685851318944845
Index: 38233
X:45037
L:1/8
M:2/4
K:G
 G>F GA | B3 G | A>A AB/A/ | G2 D2 | G>F GA | B3 G | A>A AB/A/

In [41]:
import difflib

sample = """
X:1
T:Generated Tune
M:4/4
L:1/8
K:G
 G2 | G2 G>B B>G | G>AB>G G>B | d>ef>g | B3 G | A>BA>G G>F | B>AB>A G>A |
A>GA>B B>AB>G D | E>GA>B G>B | G>B B>g G>G | A>AG>G G>B | A>GA>B G>B | d>ed>e d>G G>B |
A>BA>B F>G | A>GA>B G>B | c>AB>g G2 | A>Bc>d B>d |
""".strip()

best_length = 0
best_match = ""
best_idx = -1

for i, tune in enumerate(train_texts):
    matcher = difflib.SequenceMatcher(None, sample, tune)
    match = matcher.find_longest_match(0, len(sample), 0, len(tune))
    
    if match.size > best_length:
        best_length = match.size
        best_match = sample[match.a:match.a + match.size]
        best_idx = i

print("Longest copied substring length:", best_length)
print("Dataset index:", best_idx)
print("\nCopied substring:\n")
print(best_match)

Longest copied substring length: 14
Dataset index: 17235

Copied substring:


K:G
 G2 | G2 
